In [0]:
!pip install langchain
!pip install numpy
!pip install numpy

In [0]:
!pip install langchain_community

In [0]:
import langchain
import langchain_community

In [0]:
from langchain.chat_models import ChatDatabricks

In [0]:
# testing foundation model

chat_model = ChatDatabricks(endpoint ="databricks-claude-opus-4-5", max_tokens = 5000)
print(f"Test chat model: {chat_model.invoke('Hello, How are you today')}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat
import base64
from datetime import datetime
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatDatabricks
from docx import Document

# 1. Get notebook path from user
notebook_path = input("Enter the Databricks notebook path: ")

# 2. Fetch notebook content
def get_notebook_content(notebook_path):
    ws = WorkspaceClient()
    nb = ws.workspace.export(path=notebook_path, format=ExportFormat.SOURCE)
    return base64.b64decode(nb.content).decode("utf-8")

notebook_code = get_notebook_content(notebook_path)
generation_date = datetime.now().strftime("%Y-%m-%d")

# 3. BRD TEMPLATE (full)
TEMPLATE = """
You are a senior business analyst.

Analyze the following Azure Databricks PySpark notebook and generate a
Business Requirements Document (BRD) using the EXACT template provided below.

The output MUST be valid GitHub-flavored Markdown.

You MUST strictly follow the structure, headings, tables, and placeholders.
Replace values inside < > ONLY when they can be confidently inferred from the notebook.
If a value cannot be inferred, retain the placeholder exactly as-is.

---

Notebook Metadata:
- Source Notebook Path: {notebook_path}
- Generation Timestamp: {generation_date}

---

Notebook Content:
-----------------
{notebook_code}
-----------------

STRICT OUTPUT RULES:
- Output ONLY the BRD content in Markdown
- Do NOT include code blocks
- Do NOT include explanations or commentary
- Do NOT include any text before or after the BRD
- Do NOT add or remove sections
- Do NOT rename headings
- Preserve numbering exactly
- All tables MUST be valid Markdown tables
- Use realistic but fictional sample values where required
- If logic is directly from source, mention "n/a"
- If logic is derived, mention "calculated one"

TABLE FORMATTING RULES:
- All Markdown tables MUST have aligned columns
- Header separator rows MUST use dashes only (---)
- No extra spaces before or after pipe characters
- Each row MUST have the same number of columns

========================
BRD TEMPLATE (MANDATORY)
========================

# Business Requirements Document (BRD)

## Document Metadata

| Field | Value |
|------|-------|
| Document Name | BRD_<name of the notebook which we are reading> |
| Document Owner | <mention "dodoc.ai" here> |
| Document Version | <mention "V 1.0"> |
| Document Date | <mention current date from system here> |

---

## 1. DOCUMENT CONTROL

### 1.1 Document Audience

| Impacted Area | Stakeholder(s) | SME(s) |
|--------------|----------------|--------|
| <mention "TESTING" here> | <you can mention "Client"> | <mention "developers" here> |

---

### 1.2 Document History

| Version | Details of Change | Author | Release Date |
|--------|------------------|--------|--------------|
| 1.0 | Initial draft | <mention "dodoc.ai" here> | <mention current date from system here> |

---

### 1.3 Approval History

| Role | Name | Title | Sign-Off (Yes/No) | Date |
|-----|------|-------|------------------|------|
| QA / UAT Lead | NH | <mention "BRD Review" here> | <keep it blank> | <mention current date from system here> |
| Reviewer | AS | <mention "BRD Review" here> | <keep it blank> | <mention current date from system here> |

---

## 2. OVERVIEW AND SCOPE

### 2.1 Functional Description

| Attribute | Description | Attribute | Description |
|----------|-------------|-----------|-------------|
| Source Database | <mention "TESTING" here> | Target Database | <mention "TESTING" here> |
| Source Tables | <mention the source tables names as numbered list in seperate rows>  | Target Tables | <mention "SAMPLE" here> |
| Volumetric | <mention the count on records in the final output> | Extract Schedule | <mention "n/a" here> |
| Scope | <mention "N/A" here> | Security | <mention "N/A" here> |
| Assumptions | <mention "N/A" here> | Dependencies / Constraints | <mention "N/A" here> |

---

## 3. DETAILED FUNCTIONAL REQUIREMENTS

### 3.1 Functional Specification

<mention what the Notebook Code is about and the Business Impact here>

---

### 3.2 Filters Applied in Tables

<mention the details of filter applied on table>

---

### 3.3 Complex Logic / Transformations

<mention the details of logics we have implemented to get the output>

---

### 3.4 Calculated Columns

<mention the details of any calculated columns created with the logic details>

---

## 5. MAPPING FIELDS TO SOURCE / TARGET

<create a table as beloe provided.If transformation logic for the target column is directly from source, mention "n/a".
If it is a calculated field, mention "calculated one". 
Target Field(s) Datatype should be the datatype as per the source columns datatype and lets say if we are creating calculated column as per the data we can define it.

| Source Table(s) | Source Field(s) | Target Table(s) | Target Field(s) | Target Field(s) Datatype | Transformation Logic |
|-----------------|----------------|-----------------|------------------|--------------------------|----------------------|
"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["notebook_path", "generation_date", "notebook_code"]
)

# 4. Generate BRD markdown using LLM
chat_model = chat_model
chain = prompt | chat_model
inputs = {
    "notebook_path": notebook_path,
    "generation_date": generation_date,
    "notebook_code": notebook_code
}

brd_markdown = chain.invoke(inputs)
if hasattr(brd_markdown, "content"):
    brd_text = brd_markdown.content
else:
    brd_text = str(brd_markdown)

# 5. Ask user for desired workspace save location and file name
workspace_save_path = input("Enter the workspace folder path and file name (e.g., /Workspace/Users/your.name@company.com/your_folder/your_file.docx): ")
if not workspace_save_path.endswith('.docx'):
    workspace_save_path += '.docx'

# 6. Convert Markdown to Word (.docx) and save locally
local_tmp_path = "/tmp/temp_brd_output.docx"

def markdown_to_docx(md_text, docx_path):
    doc = Document()
    lines = md_text.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.startswith('# '):
            doc.add_heading(line[2:], level=1)
        elif line.startswith('## '):
            doc.add_heading(line[3:], level=2)
        elif line.startswith('### '):
            doc.add_heading(line[4:], level=3)
        elif line.strip().startswith('|') and line.strip().endswith('|'):
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith('|') and lines[i].strip().endswith('|'):
                table_lines.append([cell.strip() for cell in lines[i].strip().strip('|').split('|')])
                i += 1
            if table_lines:
                table = doc.add_table(rows=1, cols=len(table_lines[0]))
                hdr_cells = table.rows[0].cells
                for j, cell in enumerate(table_lines[0]):
                    hdr_cells[j].text = cell
                for row in table_lines[2:]:  # skip header and separator
                    row_cells = table.add_row().cells
                    for j, cell in enumerate(row):
                        row_cells[j].text = cell
            continue
        else:
            doc.add_paragraph(line)
        i += 1
    doc.save(docx_path)

markdown_to_docx(brd_text, local_tmp_path)

import shutil

output_filename = workspace_save_path.split('/')[-1]
dbfs_path = f"/dbfs/tmp/{output_filename}"
shutil.copyfile(local_tmp_path, dbfs_path)

from IPython.display import display, HTML
display(HTML(f'<a href="/files/tmp/{output_filename}" target="_blank">Download your BRD document: {output_filename}</a>'))
print(f"File saved to DBFS and ready for download: /dbfs/tmp/{output_filename}")



In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat
import base64
from datetime import datetime
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatDatabricks

# 1. Get notebook path from user
notebook_path = input("Enter the Databricks notebook path: ")

# 2. Fetch notebook content
def get_notebook_content(notebook_path):
    ws = WorkspaceClient()
    nb = ws.workspace.export(path=notebook_path, format=ExportFormat.SOURCE)
    return base64.b64decode(nb.content).decode("utf-8")

notebook_code = get_notebook_content(notebook_path)
generation_date = datetime.now().strftime("%Y-%m-%d")

# 3. BRD TEMPLATE (full)
TEMPLATE = """
You are a senior business analyst.

Analyze the following Azure Databricks PySpark notebook and generate a
Business Requirements Document (BRD) using the EXACT template provided below.

The output MUST be valid GitHub-flavored Markdown.

You MUST strictly follow the structure, headings, tables, and placeholders.
Replace values inside < > ONLY when they can be confidently inferred from the notebook.
If a value cannot be inferred, retain the placeholder exactly as-is.

---

Notebook Metadata:
- Source Notebook Path: {notebook_path}
- Generation Timestamp: {generation_date}

---

Notebook Content:
-----------------
{notebook_code}
-----------------

STRICT OUTPUT RULES:
- Output ONLY the BRD content in Markdown
- Do NOT include code blocks
- Do NOT include explanations or commentary
- Do NOT include any text before or after the BRD
- Do NOT add or remove sections
- Do NOT rename headings
- Preserve numbering exactly
- All tables MUST be valid Markdown tables
- Use realistic but fictional sample values where required
- If logic is directly from source, mention "n/a"
- If logic is derived, mention "calculated one"

TABLE FORMATTING RULES:
- All Markdown tables MUST have aligned columns
- Header separator rows MUST use dashes only (---)
- No extra spaces before or after pipe characters
- Each row MUST have the same number of columns

========================
BRD TEMPLATE (MANDATORY)
========================

# Business Requirements Document (BRD)

## Document Metadata

| Field | Value |
|------|-------|
| Document Name | BRD_<name of the notebook which we are reading> |
| Document Owner | <mention "dodoc.ai" here> |
| Document Version | <mention "V 1.0"> |
| Document Date | <mention current date from system here> |

---

## 1. DOCUMENT CONTROL

### 1.1 Document Audience

| Impacted Area | Stakeholder(s) | SME(s) |
|--------------|----------------|--------|
| <mention "TESTING" here> | <you can mention "Client"> | <mention "developers" here> |

---

### 1.2 Document History

| Version | Details of Change | Author | Release Date |
|--------|------------------|--------|--------------|
| 1.0 | Initial draft | <mention "dodoc.ai" here> | <mention current date from system here> |

---

### 1.3 Approval History

| Role | Name | Title | Sign-Off (Yes/No) | Date |
|-----|------|-------|------------------|------|
| QA / UAT Lead | NH | <mention "BRD Review" here> | <keep it blank> | <mention current date from system here> |
| Reviewer | AS | <mention "BRD Review" here> | <keep it blank> | <mention current date from system here> |

---

## 2. OVERVIEW AND SCOPE

### 2.1 Functional Description

| Attribute | Description | Attribute | Description |
|----------|-------------|-----------|-------------|
| Source Database | <mention "TESTING" here> | Target Database | <mention "TESTING" here> |
| Source Tables | <mention the source tables names as numbered list in seperate rows>  | Target Tables | <mention "SAMPLE" here> |
| Volumetric | <mention the count on records in the final output> | Extract Schedule | <mention "n/a" here> |
| Scope | <mention "N/A" here> | Security | <mention "N/A" here> |
| Assumptions | <mention "N/A" here> | Dependencies / Constraints | <mention "N/A" here> |

---

## 3. DETAILED FUNCTIONAL REQUIREMENTS

### 3.1 Functional Specification

<mention what the Notebook Code is about and the Business Impact here>

---

### 3.2 Filters Applied in Tables

<mention the details of filter applied on table>

---

### 3.3 Complex Logic / Transformations

<mention the details of logics we have implemented to get the output>

---

### 3.4 Calculated Columns

<mention the details of any calculated columns created with the logic details>

---

## 5. MAPPING FIELDS TO SOURCE / TARGET

<create a table as beloe provided.If transformation logic for the target column is directly from source, mention "n/a".
If it is a calculated field, mention "calculated one". 
Target Field(s) Datatype should be the datatype as per the source columns datatype and lets say if we are creating calculated column as per the data we can define it.

| Source Table(s) | Source Field(s) | Target Table(s) | Target Field(s) | Target Field(s) Datatype | Transformation Logic |
|-----------------|----------------|-----------------|------------------|--------------------------|----------------------|
"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["notebook_path", "generation_date", "notebook_code"]
)

# 4. Generate BRD markdown using LLM
chat_model = chat_model
chain = prompt | chat_model
inputs = {
    "notebook_path": notebook_path,
    "generation_date": generation_date,
    "notebook_code": notebook_code
}

# Ask for output BRD file name (for display/logging purposes)
output_filename = input("Enter desired BRD file name (for display/logging only): ")

brd_markdown = chain.invoke(inputs)
if hasattr(brd_markdown, "content"):
    brd_text = brd_markdown.content
else:
    brd_text = str(brd_markdown)

# Display the BRD content in the notebook
print(f"\n===== BRD Output: {output_filename} =====\n")
print(brd_text)
